# 07 — Quantum Advantage Scaling Benchmark

Measures wall-clock time for classical (exact diag + matrix exp) vs
quantum (Trotter on statevector) at increasing system sizes.

Classical cost scales as $O(2^{2n})$ (dense matrix expm).
Quantum cost scales as $O(n^2 \cdot r)$ gates per Trotter step.

At NISQ scale (n ≤ 14), classical wins. Exponential fit
extrapolates the crossover point where quantum becomes faster.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from scpn_quantum_control.benchmarks.quantum_advantage import (
    classical_benchmark,
    estimate_crossover,
    quantum_benchmark,
    run_scaling_benchmark,
)

## 1. Run Scaling Benchmark

Benchmark at N = 4, 6, 8, 10, 12 qubits.
N > 14 is infeasible for classical exact evolution (matrix expm).

In [ ]:
sizes = [4, 6, 8, 10, 12]
results = run_scaling_benchmark(sizes=sizes, t_max=0.5, dt=0.1)

print(f"{'N':>4} {'Classical (ms)':>15} {'Quantum (ms)':>15} {'Ratio':>8}")
print("-" * 46)
for r in results:
    ratio = r.t_classical_ms / r.t_quantum_ms if r.t_quantum_ms > 0 else float("inf")
    print(f"{r.n_qubits:4d} {r.t_classical_ms:15.2f} {r.t_quantum_ms:15.2f} {ratio:8.2f}")

if results[0].crossover_predicted:
    print(f"\nPredicted crossover: {results[0].crossover_predicted} qubits")
else:
    print("\nCrossover: not yet estimable (need more data points)")

## 2. Scaling Plot

In [ ]:
ns = [r.n_qubits for r in results]
t_c = [r.t_classical_ms for r in results]
t_q = [r.t_quantum_ms for r in results]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(ns, t_c, "o-", color="#d62728", label="Classical (expm)", linewidth=2, markersize=8)
ax.semilogy(ns, t_q, "s-", color="#1f77b4", label="Quantum (Trotter)", linewidth=2, markersize=8)

ax.set_xlabel("System size (qubits)", fontsize=12)
ax.set_ylabel("Wall-clock time (ms)", fontsize=12)
ax.set_title("Classical vs Quantum Scaling: Kuramoto XY Hamiltonian", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Classical Infeasibility Beyond N=14

The 2^N state dimension means classical expm needs ~2^(2N) operations.
At N=14, the state vector is 16384-dimensional. Beyond that, memory and time explode.

In [ ]:
for n in [12, 14, 16, 20]:
    c = classical_benchmark(n, t_max=0.5, dt=0.1)
    q = quantum_benchmark(n, t_max=0.5, dt=0.1)
    t_c_str = f"{c['t_total_ms']:.1f}" if np.isfinite(c["t_total_ms"]) else "INFEASIBLE"
    print(f"N={n:2d}: classical={t_c_str:>12} ms, quantum={q['t_total_ms']:8.1f} ms")

## 4. Crossover Extrapolation

Fit exponential curves $t(n) = a \cdot e^{bn}$ to both classical and quantum data.
If $b_c > b_q$, the curves cross at:

$$n_{\text{cross}} = \frac{\ln(a_q / a_c)}{b_c - b_q}$$

In [ ]:
from scipy.optimize import curve_fit

feasible = [r for r in results if np.isfinite(r.t_classical_ms)]
ns_f = np.array([r.n_qubits for r in feasible], dtype=float)
tc_f = np.array([r.t_classical_ms for r in feasible])
tq_f = np.array([r.t_quantum_ms for r in feasible])


def exp_model(x, a, b):
    return a * np.exp(b * x)


try:
    pc, _ = curve_fit(exp_model, ns_f, tc_f, p0=[0.01, 0.5], maxfev=2000)
    pq, _ = curve_fit(exp_model, ns_f, tq_f, p0=[0.01, 0.1], maxfev=2000)

    print(f"Classical fit: t = {pc[0]:.4f} * exp({pc[1]:.4f} * n)")
    print(f"Quantum fit:  t = {pq[0]:.4f} * exp({pq[1]:.4f} * n)")

    n_ext = np.arange(4, 40)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.semilogy(ns_f, tc_f, "o", color="#d62728", markersize=8, label="Classical (data)")
    ax.semilogy(ns_f, tq_f, "s", color="#1f77b4", markersize=8, label="Quantum (data)")
    ax.semilogy(
        n_ext, exp_model(n_ext, *pc), "--", color="#d62728", alpha=0.5, label="Classical (fit)"
    )
    ax.semilogy(
        n_ext, exp_model(n_ext, *pq), "--", color="#1f77b4", alpha=0.5, label="Quantum (fit)"
    )

    crossover = estimate_crossover(results)
    if crossover:
        ax.axvline(
            crossover, color="green", linestyle=":", linewidth=2, label=f"Crossover: n={crossover}"
        )

    ax.set_xlabel("System size (qubits)", fontsize=12)
    ax.set_ylabel("Wall-clock time (ms)", fontsize=12)
    ax.set_title("Scaling Extrapolation: Crossover Prediction", fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except RuntimeError:
    print("Curve fit failed — not enough data points for reliable extrapolation.")

## Key Takeaways

1. At N ≤ 14, classical exact evolution (matrix expm) is faster and exact.
2. Classical cost grows as O(2^2N); quantum Trotter cost grows as O(N^2 · r).
3. The crossover point depends on gate speed, error rates, and Trotter accuracy.
4. On real hardware, the crossover shifts higher due to gate errors and readout noise.
5. Quantum advantage for Kuramoto simulation requires N >> 20 with error correction.